# Imports

In [ ]:
import geopandas as gpd
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from shapely.geometry import LineString
import matplotlib.patheffects as pe
import fiona
from shapely.geometry import LineString, MultiLineString
import math

# Standardize figure formats

In [ ]:
mm = 1/25.4
mpl.rcParams.update({
    "figure.dpi": 300,          # on-screen
    "savefig.dpi": 600,         # export
    "font.family": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 6.5,           # ~6–7 pt at final size
    "axes.titlesize": 7,
    "axes.labelsize": 6.5,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
})
def add_scale_bar(ax, length_km=20, location=(0.9, 0.8), linewidth=1, tick_height=0.01, label_offset=0.02, km_offset=0.01):
    x, y = location  # Adjusted location (higher y value to move the scale bar upward)
    bar_half_length = 0.05  # Half the scale bar length in axes fraction

    # Draw the scale bar
    ax.plot(
        [x - bar_half_length, x + bar_half_length], [y, y],  # Scale bar endpoints
        transform=ax.transAxes, color='black', linewidth=linewidth
    )

    # Draw perpendicular tick marks
    tick_positions = [x - bar_half_length, x, x + bar_half_length]
    for pos in tick_positions:
        ax.plot(
            [pos, pos], [y - tick_height / 2, y + tick_height / 2],  # Vertical line for ticks
            transform=ax.transAxes, color='black', linewidth=linewidth
        )

    # Add numeric labels below the tick marks
    ax.text(
        x - bar_half_length, y - tick_height - label_offset, "0", transform=ax.transAxes, 
        ha='center', va='center', fontsize=6
    )
    ax.text(
        x, y - tick_height - label_offset, f"{length_km // 2}", transform=ax.transAxes, 
        ha='center', va='center', fontsize=6
    )
    ax.text(
        x + bar_half_length, y - tick_height - label_offset, f"{length_km}", transform=ax.transAxes, 
        ha='center', va='center', fontsize=6
    )

    # Add "km" label slightly to the right of the scale bar
    ax.text(
        x + bar_half_length + km_offset, y, "km", transform=ax.transAxes, 
        ha='left', va='center', fontsize=7
    )

def add_north_arrow(ax, location=(0.9, 0.87), size=0.05, fontsize=7, label_offset=0.03):
    """
    Add a north arrow to the plot, with "N" positioned slightly above the arrow.
    """
    x, y = location

    # Draw the arrow
    ax.annotate(
        '', xy=(x, y + size), xycoords='axes fraction',
        xytext=(x, y), textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black', headwidth=6, headlength=6, width=2.5)
    )

    # Add the "N" label slightly above the arrow
    ax.text(
        x, y + size + label_offset, "N", transform=ax.transAxes,
        fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black"
    )


In [ ]:


def longest_linestring(geom):
    if geom is None or geom.is_empty:
        return None
    if isinstance(geom, LineString):
        return geom
    if isinstance(geom, MultiLineString):
        return max(geom.geoms, key=lambda g: g.length)
    if geom.geom_type == "GeometryCollection":
        lines = [g for g in geom.geoms if g.geom_type in ("LineString", "MultiLineString")]
        return max(lines, key=lambda g: g.length) if lines else None
    return None

def angle_at_fraction(ls, frac=0.5):
    if ls is None or ls.is_empty:
        return 0.0
    pt = ls.interpolate(frac, normalized=True)
    d  = ls.interpolate(min(frac + 1e-3, 1), normalized=True)
    return math.degrees(math.atan2(d.y - pt.y, d.x - pt.x))


# Paths and CRS

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
rivers_path = gpd.read_file(base_path / "dphil_common_cross_cutting/common_incoming_data/osm/gis_osm_waterways_free_1/gis_osm_waterways_free_1.shp")

simple_rivers_path = base_path / "dphil_common_cross_cutting/common_incoming_data/rivers/rivers.gpkg"
print("GPKG in drivers?", "GPKG" in fiona.supported_drivers)
print(fiona.supported_drivers.get("GPKG"))  # should show 'rw' or 'r'

jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"

out_dir = base_path / "dphil_paper_2/results/figures/baseline_figure"

jamaica_metric_grid_crs = "EPSG:3448"

### Read in boundary 

In [ ]:
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

## Read in admin boundary 

### Read in land use

In [ ]:
land_use = gpd.read_file(base_path / "dphil_common_cross_cutting/common_incoming_data/landcover/2013_landcover/2013_landuse_LandCover.shp")
print(land_use.crs)

In [ ]:
land_use["area_m2"] = land_use.geometry.area
land_use["area_ha"] = land_use["area_m2"] / 1e4
print(f"Total area: {land_use['area_ha'].sum():,.2f} ha")

# Forest and reforestable categories

In [ ]:
# -----------------------------------------------------------
# 0)  Class sets
# -----------------------------------------------------------
forest_flood_equivalent_classes = {
    'Open dry forest - Short',
    'Open dry forest - Tall (Woodland/Savanna)',
    'Disturbed broadleaved forest (Secondary Forest)',
    'Closed broadleaved forest (Primary Forest)',
    'Secondary Forest',

}

afforestable_classes_including_agricultural = {
    'Fields: Herbaceous crops, fallow, cultivated vegetables',
    'Fields: Pasture,Human disturbed, grassland',
    'Fields: Bare Land',
    'Quarry',
    'Bauxite Extraction'
}

mixed_land_use_fractions = {
    'Fields and Secondary Forest': {
         'forest_flood_equivalent_classes': 0.50,
         'afforestable_including_agriculture': 0.50
    },
    'Bamboo and Secondary Forest': {
         'forest_flood_equivalent_classes': 1
    },
    'Bamboo and Fields': {
         'forest_flood_equivalent_classes': 0.50,
         'afforestable_including_agriculture': 0.50
    },
    'Fields  and Bamboo': {
         'forest_flood_equivalent_classes': 0.50,
         'afforestable_including_agriculture': 0.50
    },
    'Fields or Secondary Forest/Pine Plantation': {
         'forest_flood_equivalent_classes': 0.50,
         'afforestable_including_agriculture': 0.50
    }
}

# Treated-as-forest (for flood modelling) 
treated_as_forest_for_flood_purpose = {
    'Bamboo',
    'Plantation: Tree crops, shrub crops, sugar cane, banana',
    'Hardwood Plantation: Euculytus',
    'Hardwood Plantation: Mixed',
    'Hardwood Plantation: Mahoe',
    'Hardwood Plantation: Mahogany',
}


In [ ]:
class_column = "Classify"  # your field

# (optional) normalise strings to improve matching
if class_column in land_use.columns:
    land_use[class_column] = land_use[class_column].astype(str).str.strip()

# --- classifier ---
def classify_land_use(cls: str) -> str:
    if pd.isna(cls):
        return "other"
    if cls in forest_flood_equivalent_classes:
        return "existing_forest"
    if cls in treated_as_forest_for_flood_purpose:
        return "treated_forest"
    if cls in afforestable_classes_including_agricultural:
        return "reforestable"
    if cls in mixed_land_use_fractions:
        frac = mixed_land_use_fractions[cls]
        f = frac.get('forest_flood_equivalent_classes', 0)
        a = frac.get('afforestable_including_agriculture', 0)
        if f == 1 and a == 0:
            # 100% forest-like; keep as existing to avoid overusing grey
            return "existing_forest"
        if a == 1 and f == 0:
            return "reforestable"
        return "mixed"
    return "other"

land_use["forest_category"] = land_use[class_column].apply(classify_land_use)

# Plot forest and afforestable lands 

In [ ]:
# Colors
GREEN = "#2E7D32"   # existing forest
GREY  = "#757575"   # treated-as-forest (neutral mid-grey)
TAN   = "#D2B48C"   # mixed base fill
BROWN = "#654321"   # reforestable
EDGE  = "#FFFFFF"   # thin white edges

# Split by category
g_exist  = land_use[land_use["forest_category"] == "existing_forest"]
g_treat  = land_use[land_use["forest_category"] == "treated_forest"]
g_mixed  = land_use[land_use["forest_category"] == "mixed"]
g_refo   = land_use[land_use["forest_category"] == "reforestable"]
# g_other = land_use[land_use["forest_category"] == "other"]



fig, ax = plt.subplots(figsize=(180*mm, 150*mm))

# Draw order: reforestable → mixed fill → treated grey → existing green → mixed hatch
if not g_refo.empty:
    g_refo.plot(ax=ax, color=BROWN, edgecolor=EDGE, linewidth=0.15, zorder=1)
if not g_mixed.empty:
    g_mixed.plot(ax=ax, color=TAN, edgecolor=EDGE, linewidth=0.15, zorder=2)
if not g_treat.empty:
    g_treat.plot(ax=ax, color=GREY, edgecolor=EDGE, linewidth=0.15, zorder=3)
if not g_exist.empty:
    g_exist.plot(ax=ax, color=GREEN, edgecolor=EDGE, linewidth=0.15, zorder=4)

# Hatch overlay for mixed (green slashes)
if not g_mixed.empty:
    g_mixed.plot(ax=ax, facecolor="none", edgecolor=GREEN, hatch="///", linewidth=0.2, zorder=5)


# draw a white halo, then a dark stroke on top
jamaica_boundary.plot(ax=ax, facecolor="none", edgecolor="white",
                      linewidth=1.2, zorder=98)
jamaica_boundary.plot(ax=ax, facecolor="none", edgecolor="#222222",
                     linewidth=0.6, zorder=99)   # <-- completed line

ax.set_axis_off()
ax.set_title("Fig. 1: Map of land use categories to inform forest restoration planning for river flood alleviation", pad=6)

legend_handles = [
    Patch(facecolor=GREEN, edgecolor="black", linewidth=0.3, label="Existing forest"),
    Patch(facecolor=GREY,  edgecolor="black", linewidth=0.3, label="Treated as forest for flood modelling"),
    Patch(facecolor=TAN,   edgecolor=GREEN,  hatch="///", linewidth=0.3, label="Half existing forest / half restorable"),
    Patch(facecolor=BROWN, edgecolor="black", linewidth=0.3, label="Candidates for forest restoration"),
]
leg = ax.legend(handles=legend_handles, loc="lower left", frameon=True, framealpha=0.9, fontsize=6)
leg.get_frame().set_linewidth(0.4)



add_scale_bar(ax)
add_north_arrow(ax)

plt.tight_layout()

# where to save
fname = out_dir / "landuse_categories"

# save BEFORE plt.show()
plt.savefig(fname.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")  # print-ready raster

plt.show()
print("Saved to:", fname.with_suffix(".png"))

In [ ]:
# === Split "existing_forest" into forest subtypes and plot with shades of green ===

# 1) Create a forest subtype column only for true forest classes
def existing_forest_subtype(cls: str):
    return cls if cls in forest_flood_equivalent_classes else None

land_use["forest_subtype"] = land_use[class_column].apply(existing_forest_subtype)

# 2) Palette for forest subtypes (ordered darkest→lightest)
forest_palette_fixed = {
    "Closed broadleaved forest (Primary Forest)"      : "#1B5E20",
    "Disturbed broadleaved forest (Secondary Forest)" : "#32CD32",
    "Secondary Forest"                                : "#388E3C",
    "Open dry forest - Tall (Woodland/Savanna)"       : "#6B8E23",
    "Open dry forest - Short"                         : "#9ACD32",
}

# Subtypes present
present_subtypes = [t for t in forest_flood_equivalent_classes
                    if (land_use["forest_subtype"] == t).any()]
missing = [t for t in present_subtypes if t not in forest_palette_fixed]
if missing:
    import matplotlib.colors as mcolors
    cmap = plt.cm.get_cmap("Greens", len(missing) + 3)
    for i, t in enumerate(missing, start=1):
        forest_palette_fixed[t] = mcolors.to_hex(cmap(i))

# 3) Convenience subsets
g_treat = land_use[land_use["forest_category"] == "treated_forest"]
g_mixed = land_use[land_use["forest_category"] == "mixed"]
g_refo  = land_use[land_use["forest_category"] == "reforestable"]

# Special case subset (do NOT plot yet)
MIXED_SEC_LABEL = "Bamboo and Secondary Forest"
g_mixed_sec = land_use[land_use[class_column].eq(MIXED_SEC_LABEL)]

# 4) Figure
fig, ax = plt.subplots(figsize=(180*mm, 150*mm))

# Base layers
if not g_refo.empty:
    g_refo.plot(ax=ax, color=BROWN, edgecolor=EDGE, linewidth=0.15, zorder=1)
if not g_mixed.empty:
    g_mixed.plot(ax=ax, color=TAN, edgecolor=EDGE, linewidth=0.15, zorder=2)
if not g_treat.empty:
    g_treat.plot(ax=ax, color=GREY, edgecolor=EDGE, linewidth=0.15, zorder=3)

# 4a) Special case: Bamboo + Secondary Forest as its own category (grey + green hatch)
if not g_mixed_sec.empty:
    g_mixed_sec.plot(ax=ax, color=GREY, edgecolor=EDGE, linewidth=0.15, zorder=3.5)
    g_mixed_sec.plot(ax=ax, facecolor="none", edgecolor=GREEN, hatch="///", linewidth=0.2, zorder=91)

# 5) Forest subtypes
z = 4
for subtype in present_subtypes:
    gs = land_use[land_use["forest_subtype"] == subtype]
    if not gs.empty:
        color = forest_palette_fixed[subtype]
        gs.plot(ax=ax, color=color, edgecolor=EDGE, linewidth=0.15, zorder=z)
        z += 1

# Hatch overlay for generic mixed
if not g_mixed.empty:
    g_mixed.plot(ax=ax, facecolor="none", edgecolor=GREEN, hatch="///", linewidth=0.2, zorder=90)

# Outline
jamaica_boundary.plot(ax=ax, facecolor="none", edgecolor="white",   linewidth=1.2, zorder=98)
jamaica_boundary.plot(ax=ax, facecolor="none", edgecolor="#222222", linewidth=0.6, zorder=99)

ax.set_axis_off()
ax.set_title("")

# --- Ordered legend ---
legend_label = {
    "Closed broadleaved forest (Primary Forest)"      : "Closed broadleaved forest (Primary Forest)",
    "Secondary Forest"                                : "Secondary Forest",
    "Disturbed broadleaved forest (Secondary Forest)" : "Disturbed broadleaved forest (Secondary Forest)",
    "Open dry forest - Tall (Woodland/Savanna)"       : "Open dry forest (tall)",
    "Open dry forest - Short"                         : "Open dry forest (short)",
    "treated_forest"                                  : "Treated as forest for flood modelling",
    "reforestable"                                    : "Candidates for forest restoration",
    "mixed"                                           : "Half existing forest / half restorable",
    "mixed_secondary_treated"                         : "Mixed secondary forest / treated as forest",
}

legend_order = [
    "Closed broadleaved forest (Primary Forest)",
    "Secondary Forest",
    "Disturbed broadleaved forest (Secondary Forest)",
    "Open dry forest - Tall (Woodland/Savanna)",
    "Open dry forest - Short",
    "treated_forest",
    "mixed_secondary_treated",
    "reforestable",
    "mixed",
]

from matplotlib.patches import Patch
handles_by_key = {}

# forest subtype handles
for subtype in present_subtypes:
    if (land_use["forest_subtype"] == subtype).any():
        handles_by_key[subtype] = Patch(
            facecolor=forest_palette_fixed[subtype],
            edgecolor="black", linewidth=0.3,
            label=legend_label[subtype]
        )

# other category handles
if not g_treat.empty:
    handles_by_key["treated_forest"] = Patch(facecolor=GREY, edgecolor="black", linewidth=0.3,
                                             label=legend_label["treated_forest"])
if not g_refo.empty:
    handles_by_key["reforestable"] = Patch(facecolor=BROWN, edgecolor="black", linewidth=0.3,
                                           label=legend_label["reforestable"])
if not g_mixed.empty:
    handles_by_key["mixed"] = Patch(facecolor=TAN, edgecolor=GREEN, hatch="///", linewidth=0.3,
                                    label=legend_label["mixed"])
if not g_mixed_sec.empty:
    handles_by_key["mixed_secondary_treated"] = Patch(facecolor=GREY, edgecolor=GREEN, hatch="///", linewidth=0.3,
                                                      label=legend_label["mixed_secondary_treated"])

ordered_handles = [handles_by_key[k] for k in legend_order if k in handles_by_key]
leg = ax.legend(handles=ordered_handles, loc="lower left",
                frameon=True, framealpha=0.9, fontsize=5, ncol=1,
                title="Land use categories", title_fontsize=6)
leg.get_frame().set_linewidth(0.4)

# North arrow & scale bar
add_scale_bar(ax, location=(0.88, 0.78), length_km=20, linewidth=0.6, label_offset=0.02, km_offset=0.01)
add_north_arrow(ax, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)

plt.tight_layout()

# Save
fname = out_dir / "Land_use_categories_altered_legend"
plt.savefig(fname.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved to:", fname.with_suffix(".png"))

# AREA CALCULATIONS

In [ ]:
# --- Assumes: land_use in a metric CRS; class_column defined;
#              afforestable_classes_including_agricultural, mixed_land_use_fractions,
#              forest_flood_equivalent_classes defined ---

# Ensure out_dir exists (adjust base_path if needed)
try:
    out_dir
except NameError:
    base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
    out_dir = (base_path / "Outputs")
out_dir.mkdir(parents=True, exist_ok=True)

# 1) Weights
refo_weights = {
    **{k: 1.0 for k in afforestable_classes_including_agricultural},
    **{k: v.get("afforestable_including_agriculture", 0.0)
       for k, v in mixed_land_use_fractions.items()}
}
exist_weights = {
    **{k: 1.0 for k in forest_flood_equivalent_classes},
    **{k: v.get("forest_flood_equivalent_classes", 0.0)
       for k, v in mixed_land_use_fractions.items()}
}

# 2) Areas
lu = land_use.copy()
lu["_area_m2"] = lu.geometry.area

# 3) Totals
treated_labels = {"treated_forest"}  # use only labels you actually have
reforest_total_m2      = float((lu["_area_m2"] * lu[class_column].map(refo_weights).fillna(0.0)).sum())
treated_area_m2        = float(lu.loc[lu["forest_category"].isin(treated_labels), "_area_m2"].sum())
existing_pure_m2       = float(lu.loc[lu["forest_category"].eq("existing_forest"), "_area_m2"].sum())
existing_weighted_m2   = float((lu["_area_m2"] * lu[class_column].map(exist_weights).fillna(0.0)).sum())
combined_total_m2      = reforest_total_m2 + treated_area_m2

# 4) Export table
area_totals = pd.DataFrame(
    [
        {"metric": "reforestable_weighted",    "area_m2": reforest_total_m2},
        {"metric": "treated_as_forest",        "area_m2": treated_area_m2},
        {"metric": "combined",                 "area_m2": combined_total_m2},
        {"metric": "existing_forest_pure",     "area_m2": existing_pure_m2},
        {"metric": "existing_forest_weighted", "area_m2": existing_weighted_m2},
    ]
).assign(
    area_ha=lambda d: d.area_m2 / 1e4,
    area_km2=lambda d: d.area_m2 / 1e6,
)

# 5) Save (both formats)
parquet_path = out_dir / "forest_area_totals.parquet"
csv_path     = out_dir / "forest_area_totals.csv"
area_totals.to_parquet(parquet_path, index=False)
area_totals.to_csv(csv_path, index=False)

print("Wrote:", parquet_path)
print("Wrote:", csv_path)

# River info

In [ ]:
rivers_simple = gpd.read_file(simple_rivers_path)  # add layer=... if needed


In [ ]:
# --- labels dataframe ---
labels = (
    rivers_simple.loc[rivers_simple["name"].notna() & (rivers_simple["name"].str.strip() != "")]
    .assign(len_m=lambda df: df.geometry.length)
    .sort_values("len_m", ascending=False)
    .drop_duplicates(subset="name")
    .copy()
)
labels["geom_ls"]     = labels.geometry.apply(longest_linestring)
labels["label_point"] = labels["geom_ls"].apply(lambda g: g.interpolate(0.5, normalized=True))
labels["angle"]       = labels["geom_ls"].apply(angle_at_fraction)
labels = labels[labels["geom_ls"].apply(lambda g: g.length >= 1500)].copy()

# --- plot ---
fig, ax = plt.subplots(figsize=(10, 10))

jamaica_boundary.boundary.plot(ax=ax, color="black", linewidth=0.8, zorder=1)

rivers_simple.plot(ax=ax, linewidth=0.6, color="#1f77b4", zorder=2)

for _, r in labels.iterrows():
    p = r["label_point"]
    ax.text(p.x, p.y, r["name"], fontsize=8,
            rotation=r["angle"], rotation_mode="anchor",
            ha="center", va="center", zorder=3,
            path_effects=[pe.withStroke(linewidth=2, foreground="white")])

# tighten to boundary so scalebar sizes sensibly
minx, miny, maxx, maxy = jamaica_boundary.total_bounds
ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy)

add_scale_bar(ax)
add_north_arrow(ax)
title = "Rivers"  # pick your title

ax.set_axis_off()
plt.title(title, fontsize=20, fontweight='bold',
          fontname='Times New Roman', pad=20)
plt.tight_layout()

out_png = Path(out_dir) /"rivers.png"
fig.savefig(out_png, dpi=300, bbox_inches='tight')
print(f"Saved {out_png}")
plt.show()
ax.set_title("Simple Rivers with Labels, Jamaica")
ax.set_axis_off()
plt.tight_layout()
plt.show()